# AI Agent for Advertisement Auditing

This project audits live advertisements against a product catalogue and claims policy.  
Each advertisement is classified as `pass`, `auto_fix`, or `escalate`.

The agent checks:

- Product availability and status
- Prohibited advertising claims
- Attribute-backed claims
- Advertised price accuracy

In [12]:
import pandas as pd
!pip install pandas

In [13]:
from google.colab import files

uploaded = files.upload()

Saving catalogue.csv to catalogue (1).csv
Saving ads.csv to ads (1).csv
Saving claims_policy.md to claims_policy (1).md


In [14]:
catalogue = pd.read_csv("catalogue.csv")
ads = pd.read_csv("ads.csv")

print(f"Catalogue records: {len(catalogue)}")
print(f"Advertisement records: {len(ads)}")

Catalogue records: 300
Advertisement records: 250


In [15]:
print("Catalogue sample:")
display(catalogue.head())

print("\nAdvertisement sample:")
display(ads.head())

Catalogue sample:


,sku,brand,product_name,category,price,status,material,attributes
0,SKU-1112,Aeris,Aeris Collagen,Supplements,19.9,discontinued,stainless steel,bpa_free;fragrance_free
1,SKU-1173,Lumen,Lumen Bluetooth Speaker,Electronics,49.0,active,stainless steel,clinically_tested;fragrance_free
2,SKU-1219,Aeris,Aeris Air Purifier,Home,14.5,active,organic cotton,fragrance_free;hypoallergenic;water_resistant
3,SKU-1140,Tenzo,Tenzo Protein Powder,Supplements,59.0,active,stainless steel,clinically_tested;hypoallergenic;water_resistant
4,SKU-1297,Vela,Vela Sports Bra,Apparel,129.0,active,organic cotton,NaN



Advertisement sample:


,ad_id,sku,headline,advertised_price
0,AD-5001,SKU-1313,Mira Trainers — SPF 50 protection,37.0
1,AD-5002,SKU-1012,Lumen Moisturiser — prevents COVID-19,51.1
2,AD-5003,SKU-1157,Shop the Mira Power Bank,35.9
3,AD-5004,SKU-1095,Shop the Sable Probiotic,134.0
4,AD-5005,SKU-1042,Nuvo Toner — hypoallergenic formula,116.1


In [16]:
print("Catalogue columns:")
print(catalogue.columns.tolist())

print("\nAdvertisement columns:")
print(ads.columns.tolist())

print("\nMissing values in catalogue:")
print(catalogue.isnull().sum())

print("\nMissing values in ads:")
print(ads.isnull().sum())

Catalogue columns:
['sku', 'brand', 'product_name', 'category', 'price', 'status', 'material', 'attributes']

Advertisement columns:
['ad_id', 'sku', 'headline', 'advertised_price']

Missing values in catalogue:
sku              0
brand            0
product_name     0
category         0
price            0
status           0
material         0
attributes      74
dtype: int64

Missing values in ads:
ad_id               0
sku                 0
headline            0
advertised_price    0
dtype: int64


In [17]:
# Policy rules

PROHIBITED_CLAIMS = [
    "cures eczema",
    "prevents covid-19",
    "guaranteed weight loss",
    "doctor recommended #1",
    "reverses ageing"
]

ATTRIBUTE_CLAIMS = {
    "100% waterproof": "waterproof",
    "spf 50 protection": "spf50",
    "clinically proven": "clinically_tested",
    "bpa-free": "bpa_free",
    "active noise cancelling": "noise_cancelling",
    "hypoallergenic formula": "hypoallergenic",
    "100% vegan": "vegan",
    "fragrance-free": "fragrance_free"
}

print(f"Prohibited claims: {len(PROHIBITED_CLAIMS)}")
print(f"Attribute-backed claims: {len(ATTRIBUTE_CLAIMS)}")

Prohibited claims: 5
Attribute-backed claims: 8


In [18]:
def parse_attributes(value):
    if pd.isna(value):
        return set()

    return {
        attribute.strip().lower()
        for attribute in str(value).split(";")
        if attribute.strip()
    }


def audit_ad(ad, catalogue_lookup):
    ad_id = ad["ad_id"]
    sku = ad["sku"]
    headline = str(ad["headline"]).strip()
    headline_lower = headline.lower()
    advertised_price = float(ad["advertised_price"])

    if sku not in catalogue_lookup.index:
        return {
            "ad_id": ad_id,
            "decision": "escalate",
            "corrected_value": "",
            "reason": f"SKU {sku} was not found in the product catalogue."
        }

    product = catalogue_lookup.loc[sku]

    catalogue_price = float(product["price"])
    product_status = str(product["status"]).strip().lower()
    product_attributes = parse_attributes(product["attributes"])

    if product_status == "discontinued":
        return {
            "ad_id": ad_id,
            "decision": "escalate",
            "corrected_value": "",
            "reason": "The product is discontinued and requires human review."
        }

    prohibited_claim = next(
        (
            claim
            for claim in PROHIBITED_CLAIMS
            if claim in headline_lower
        ),
        None
    )

    if prohibited_claim:
        return {
            "ad_id": ad_id,
            "decision": "escalate",
            "corrected_value": "",
            "reason": (
                f'The headline contains the prohibited claim '
                f'"{prohibited_claim}".'
            )
        }

    for claim_phrase, required_attribute in ATTRIBUTE_CLAIMS.items():
        if (
            claim_phrase in headline_lower
            and required_attribute not in product_attributes
        ):
            return {
                "ad_id": ad_id,
                "decision": "escalate",
                "corrected_value": "",
                "reason": (
                    f'The claim "{claim_phrase}" is unsupported because '
                    f'the product does not have the '
                    f'"{required_attribute}" attribute.'
                )
            }

    if round(advertised_price, 2) != round(catalogue_price, 2):
        return {
            "ad_id": ad_id,
            "decision": "auto_fix",
            "corrected_value": round(catalogue_price, 2),
            "reason": (
                f"The advertised price {advertised_price:.2f} does not "
                f"match the catalogue price {catalogue_price:.2f}."
            )
        }

    return {
        "ad_id": ad_id,
        "decision": "pass",
        "corrected_value": "",
        "reason": (
            "The product is active, the price matches, and no prohibited "
            "or unsupported claim was found."
        )
    }

In [19]:
# Build catalogue lookup for fast SKU retrieval

catalogue_lookup = catalogue.set_index("sku")

results = []

for _, ad in ads.iterrows():
    result = audit_ad(ad, catalogue_lookup)
    results.append(result)

results_df = pd.DataFrame(results)

print("Audit completed successfully.")
print(f"Total ads processed: {len(results_df)}")

Audit completed successfully.
Total ads processed: 250


In [20]:
# Display first 10 results

display(results_df.head(10))

,ad_id,decision,corrected_value,reason
0,AD-5001,escalate,,"The claim ""spf 50 protection"" is unsupported b..."
1,AD-5002,escalate,,"The headline contains the prohibited claim ""pr..."
2,AD-5003,auto_fix,39.0,The advertised price 35.90 does not match the ...
3,AD-5004,pass,,"The product is active, the price matches, and ..."
4,AD-5005,escalate,,"The claim ""hypoallergenic formula"" is unsuppor..."
5,AD-5006,escalate,,The product is discontinued and requires human...
6,AD-5007,pass,,"The product is active, the price matches, and ..."
7,AD-5008,escalate,,"The headline contains the prohibited claim ""re..."
8,AD-5009,pass,,"The product is active, the price matches, and ..."
9,AD-5010,pass,,"The product is active, the price matches, and ..."


In [21]:
# Display decision counts

decision_summary = (
    results_df["decision"]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="count")
)

display(decision_summary)

,decision,count
0,pass,110
1,escalate,90
2,auto_fix,50


In [22]:
expected_columns = [
    "ad_id",
    "decision",
    "corrected_value",
    "reason"
]

valid_decisions = {
    "pass",
    "auto_fix",
    "escalate"
}

assert results_df.columns.tolist() == expected_columns
assert len(results_df) == len(ads)
assert results_df["ad_id"].is_unique
assert results_df["decision"].isin(valid_decisions).all()

assert (
    results_df.loc[
        results_df["decision"] != "auto_fix",
        "corrected_value"
    ]
    .fillna("")
    .eq("")
    .all()
)

print("All output validation checks passed.")

All output validation checks passed.


In [23]:
print("Pass examples:")
display(results_df[results_df["decision"] == "pass"].head(3))

print("\nAuto-fix examples:")
display(results_df[results_df["decision"] == "auto_fix"].head(3))

print("\nEscalation examples:")
display(results_df[results_df["decision"] == "escalate"].head(3))

Pass examples:


,ad_id,decision,corrected_value,reason
3,AD-5004,pass,,"The product is active, the price matches, and ..."
6,AD-5007,pass,,"The product is active, the price matches, and ..."
8,AD-5009,pass,,"The product is active, the price matches, and ..."



Auto-fix examples:


,ad_id,decision,corrected_value,reason
2,AD-5003,auto_fix,39.0,The advertised price 35.90 does not match the ...
15,AD-5016,auto_fix,139.0,The advertised price 149.00 does not match the...
16,AD-5017,auto_fix,24.0,The advertised price 34.00 does not match the ...



Escalation examples:


,ad_id,decision,corrected_value,reason
0,AD-5001,escalate,,"The claim ""spf 50 protection"" is unsupported b..."
1,AD-5002,escalate,,"The headline contains the prohibited claim ""pr..."
4,AD-5005,escalate,,"The claim ""hypoallergenic formula"" is unsuppor..."


In [26]:
results_df.to_csv("results.csv", index=False)
print("results.csv has been created successfully.")

results.csv has been created successfully.


In [27]:
from google.colab import files

# Download results
files.download("results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>